# QB (Quarterback) Round Regression (Ridge)

Predict draft round 1–8 (8 = undrafted) for quarterbacks using combine + RAS + PFF passing efficiency metrics.

- **Train**: 2015–2023 from `qb_training.csv`
- **Test**: `qb_testing.csv` filtered to 2024/2025 (drafted only); 2026 predictions for all prospects.


In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

FEATURES_WITH_COLLEGE_QB = [
    # Physical / Athletic
    'Height', 'Weight', '40yd', 'speed_score', 'RAS',
    # Passing – Decision Making
    'btt_rate', 'twp_rate',
    # Passing – Efficiency
    'ypa', 'qb_rating',
    # Passing – Pressure / Sack Profile
    'pressure_to_sack_rate', 'sack_percent',
    # Passing – Advanced Efficiency
    'epa', 'positive_epa_percent',
    # Passing – Style
    'avg_depth_of_target',
    # Context / Experience
    'player_game_count', 'p4_conference',
]

CONTAINS_WITH_COLLEGE_QB = [
    'contains_height', 'contains_weight', 'contains_40yd',
    'contains_speed_score', 'contains_ras',
    'contains_btt_rate', 'contains_twp_rate',
    'contains_ypa', 'contains_qb_rating',
    'contains_pressure_to_sack_rate', 'contains_sack_percent',
    'contains_epa', 'contains_positive_epa_percent',
    'contains_avg_depth_of_target',
    'contains_player_game_count', 'contains_p4_conference',
]

FEATURES_ALL = FEATURES_WITH_COLLEGE_QB + CONTAINS_WITH_COLLEGE_QB

In [2]:
# ── P4 conference sets ──────────────────────────────────────────────────────
P4_PRE_2024 = {
    # SEC
    'Alabama', 'Arkansas', 'Auburn', 'Florida', 'Georgia', 'Kentucky',
    'LSU', 'Mississippi', 'Mississippi State', 'Missouri', 'South Carolina',
    'Tennessee', 'Texas A&M', 'Vanderbilt',
    # Big Ten
    'Illinois', 'Indiana', 'Iowa', 'Maryland', 'Michigan', 'Michigan State',
    'Minnesota', 'Nebraska', 'Northwestern', 'Ohio State', 'Penn State',
    'Purdue', 'Rutgers', 'Wisconsin',
    # Big 12
    'Baylor', 'Iowa State', 'Kansas', 'Kansas State', 'Oklahoma',
    'Oklahoma State', 'TCU', 'Texas', 'Texas Tech', 'West Virginia',
    'Cincinnati', 'Houston', 'UCF', 'BYU',
    # ACC
    'Boston College', 'Clemson', 'Duke', 'Florida State', 'Georgia Tech',
    'Louisville', 'Miami', 'North Carolina', 'North Carolina State',
    'Pittsburgh', 'Syracuse', 'Virginia', 'Virginia Tech', 'Wake Forest',
    # Pac-12 (valid thru 2023)
    'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon',
    'Oregon State', 'Stanford', 'UCLA', 'USC', 'Utah', 'Washington',
    'Washington State',
}

P4_2024_PLUS = P4_PRE_2024 - {
    'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon', 'Oregon State',
    'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State',
} | {
    # Oregon, Washington, UCLA, USC moved to Big Ten 2024
    'Oregon', 'Washington', 'UCLA', 'USC',
    # Arizona, Arizona St, Utah, Colorado moved to Big 12 2024
    'Arizona', 'Arizona State', 'Utah', 'Colorado',
    # SMU joined ACC 2024
    'SMU',
    # Stanford, Cal, Washington St, Oregon St still in diminished Pac-12 / went independent
    # (NOT counted as P4 post-2024)
}
# For simplicity keep Oregon State, Washington State, California, Stanford out of P4 2024+
P4_2024_PLUS -= {'Oregon State', 'Washington State', 'California', 'Stanford'}


def get_p4(school, year):
    ref = P4_PRE_2024 if year < 2024 else P4_2024_PLUS
    return 1 if str(school).strip() in ref else 0


def add_engineered_features(df):
    """Add speed_score, p4_conference, and contains_* flags."""
    df = df.copy()
    # Height: convert 'feet-inches' string if needed
    if df['Height'].dtype == object or df['Height'].astype(str).str.contains('-', na=False).any():
        def _ht(h):
            if pd.isna(h): return np.nan
            s = str(h).strip()
            if '-' in s:
                p = s.split('-')
                try: return int(p[0]) * 12 + int(p[1])
                except: return np.nan
            try: return float(s)
            except: return np.nan
        df['Height'] = df['Height'].apply(_ht)
    else:
        df['Height'] = pd.to_numeric(df['Height'], errors='coerce')

    for c in ['Weight', '40yd', 'RAS', 'arm_length_inches',
              'btt_rate', 'twp_rate', 'ypa', 'qb_rating',
              'pressure_to_sack_rate', 'sack_percent', 'epa',
              'positive_epa_percent', 'avg_depth_of_target', 'player_game_count']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # speed_score
    w = pd.to_numeric(df['Weight'], errors='coerce')
    s = pd.to_numeric(df['40yd'], errors='coerce')
    df['speed_score'] = np.where((s > 0) & (~w.isna()), w * 200 / (s ** 4), np.nan)

    # p4_conference
    df['p4_conference'] = df.apply(lambda r: get_p4(r['School'], int(r['Year'])), axis=1)

    # contains_* flags
    df['contains_height']           = df['Height'].notna().astype(int)
    df['contains_weight']           = df['Weight'].notna().astype(int)
    df['contains_40yd']             = df['40yd'].notna().astype(int)
    df['contains_speed_score']      = df['speed_score'].notna().astype(int)
    df['contains_ras']              = df['RAS'].notna().astype(int) if 'RAS' in df.columns else 0
    df['contains_btt_rate']         = df['btt_rate'].notna().astype(int) if 'btt_rate' in df.columns else 0
    df['contains_twp_rate']         = df['twp_rate'].notna().astype(int) if 'twp_rate' in df.columns else 0
    df['contains_ypa']              = df['ypa'].notna().astype(int) if 'ypa' in df.columns else 0
    df['contains_qb_rating']        = df['qb_rating'].notna().astype(int) if 'qb_rating' in df.columns else 0
    df['contains_pressure_to_sack_rate'] = df['pressure_to_sack_rate'].notna().astype(int) if 'pressure_to_sack_rate' in df.columns else 0
    df['contains_sack_percent']     = df['sack_percent'].notna().astype(int) if 'sack_percent' in df.columns else 0
    df['contains_epa']              = df['epa'].notna().astype(int) if 'epa' in df.columns else 0
    df['contains_positive_epa_percent'] = df['positive_epa_percent'].notna().astype(int) if 'positive_epa_percent' in df.columns else 0
    df['contains_avg_depth_of_target'] = df['avg_depth_of_target'].notna().astype(int) if 'avg_depth_of_target' in df.columns else 0
    df['contains_player_game_count'] = df['player_game_count'].notna().astype(int) if 'player_game_count' in df.columns else 0
    df['contains_p4_conference']    = 1  # always available

    return df

In [3]:
# ── Load and prepare training data ──────────────────────────────────────────
df = pd.read_csv('../data/processed/qb_training.csv')
df = df[df['Year'].between(2015, 2023)].copy()
print(f'Train (2015–2023 QBs): {len(df)}')

df = add_engineered_features(df)

# Add missing feature columns with 0
for c in FEATURES_ALL:
    if c not in df.columns:
        df[c] = 0

# Target: round 1–8 (8 = undrafted)
y = np.where(df['Drafted'].astype(bool), np.clip(df['Round'].fillna(1).astype(int), 1, 7), 8)
X_raw = df[FEATURES_ALL].copy()

imputer = KNNImputer(n_neighbors=10)
X = imputer.fit_transform(X_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_scaled, y)

y_pred_train = np.clip(ridge.predict(X_scaled), 1, 8)
from sklearn.metrics import mean_absolute_error
print(f'Train MAE (round 1–8): {mean_absolute_error(y, y_pred_train):.4f}')
print(f'Train samples: {len(y)}')

Train (2015–2023 QBs): 148
Train MAE (round 1–8): 1.8087
Train samples: 148


In [4]:
# ── Data availability summary ─────────────────────────────────────────────
total = len(df)
for feat in FEATURES_WITH_COLLEGE_QB:
    base = feat.replace('contains_', '')
    col = base if base in df.columns else None
    if col:
        n = df[col].notna().sum()
        print(f'{col}: {n}/{total} ({100*n/total:.0f}%)')

Height: 147/148 (99%)
Weight: 147/148 (99%)
40yd: 118/148 (80%)
speed_score: 117/148 (79%)
RAS: 83/148 (56%)
btt_rate: 129/148 (87%)
twp_rate: 129/148 (87%)
ypa: 129/148 (87%)
qb_rating: 129/148 (87%)
pressure_to_sack_rate: 129/148 (87%)
sack_percent: 129/148 (87%)
epa: 129/148 (87%)
positive_epa_percent: 129/148 (87%)
avg_depth_of_target: 129/148 (87%)
player_game_count: 129/148 (87%)
p4_conference: 148/148 (100%)


In [5]:
# ── Load testing data ─────────────────────────────────────────────────────
qb_testing = pd.read_csv('../data/processed/qb_testing.csv')

qb_2024 = qb_testing[(qb_testing['Year'] == 2024) & (pd.to_numeric(qb_testing['Round'], errors='coerce') < 8)].copy()
qb_2025 = qb_testing[(qb_testing['Year'] == 2025) & (pd.to_numeric(qb_testing['Round'], errors='coerce') < 8)].copy()
qb_2026 = qb_testing[qb_testing['Year'] == 2026].copy()

print(f'QB 2024 drafted: {len(qb_2024)}')
print(f'QB 2025 drafted: {len(qb_2025)}')
print(f'QB 2026 prospects: {len(qb_2026)}')


def prepare_qb_df(ldf, year):
    ldf = ldf.copy()
    ldf['Year'] = year
    ldf = add_engineered_features(ldf)
    for c in FEATURES_ALL:
        if c not in ldf.columns:
            ldf[c] = 0
    return ldf


def eval_metrics(actual, pred, label):
    mae   = mean_absolute_error(actual, pred)
    rmse  = np.sqrt(mean_squared_error(actual, pred))
    r2    = r2_score(actual, pred)
    exact = (np.round(pred) == actual).mean()
    w1    = (np.abs(np.round(pred) - actual) <= 1).mean()
    print(f'{label} (n={len(actual)}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, '
          f'Exact={exact:.2%}, Within-1={w1:.2%}')

QB 2024 drafted: 11
QB 2025 drafted: 11
QB 2026 prospects: 16


In [6]:
# ── Evaluate 2024 and 2025 ────────────────────────────────────────────────
qb_2024 = prepare_qb_df(qb_2024, 2024)
qb_2025 = prepare_qb_df(qb_2025, 2025)

X_24 = scaler.transform(imputer.transform(qb_2024[FEATURES_ALL]))
X_25 = scaler.transform(imputer.transform(qb_2025[FEATURES_ALL]))

pred_24 = np.clip(ridge.predict(X_24), 1, 8)
pred_25 = np.clip(ridge.predict(X_25), 1, 8)

actual_24 = pd.to_numeric(qb_2024['Round'], errors='coerce').fillna(8).astype(int).values
actual_25 = pd.to_numeric(qb_2025['Round'], errors='coerce').fillna(8).astype(int).values

eval_metrics(actual_24, pred_24, '2024 QBs')
eval_metrics(actual_25, pred_25, '2025 QBs')

2024 QBs (n=11): MAE=1.4618, RMSE=1.7739, R²=0.5816, Exact=27.27%, Within-1=36.36%
2025 QBs (n=11): MAE=2.5436, RMSE=2.9114, R²=-1.1104, Exact=9.09%, Within-1=18.18%


In [7]:
# ── 2024 drafted QBs — predictions table ────────────────────────────────
def qb_tier(p):
    if p < 1.75: return ('Round 1', 'True 1st-round QB profile')
    if p < 2.75: return ('Round 2', 'Early Day 2 QB')
    if p < 3.75: return ('Round 3', 'Late Day 2 QB')
    if p < 4.75: return ('Round 4', 'Early Day 3 QB')
    if p < 5.75: return ('Round 5', 'Mid Day 3 QB')
    if p < 6.75: return ('Round 6', 'Late Day 3 QB')
    if p < 7.75: return ('Round 7', 'Fringe draftable QB')
    return ('UDFA', 'Undrafted')

d24_display = qb_2024[['Round', 'Pick', 'Player', 'School']].copy()
d24_display['predicted_round'] = pred_24
d24_display['tier'] = [qb_tier(x)[0] for x in pred_24]
d24_display['Round'] = pd.to_numeric(d24_display['Round'], errors='coerce').astype('Int64')
print('2024 drafted QBs')
display(d24_display.sort_values('predicted_round').reset_index(drop=True))

2024 drafted QBs


,Round,Pick,Player,School,predicted_round,tier
0,1,2.00,Jayden Daniels,LSU,1.00,Round 1
1,1,10.00,J.J. McCarthy,Michigan,1.00,Round 1
2,1,12.00,Bo Nix,Oregon,1.00,Round 1
3,1,1.00,Caleb Williams,USC,2.73,Round 2
4,1,3.00,Drake Maye,North Carolina,2.73,Round 2
5,1,8.00,Michael Penix Jr.,Washington,2.92,Round 3
6,7,245.00,Michael Pratt,Tulane,3.68,Round 3
7,6,171.00,Jordan Travis,Florida State,4.21,Round 4
8,7,193.00,Joe Milton,Tennessee,4.80,Round 5
9,7,218.00,Devin Leary,Kentucky,5.56,Round 5


In [8]:
# ── 2025 drafted QBs — predictions table ────────────────────────────────
d25_display = qb_2025[['Round', 'Pick', 'Player', 'School']].copy()
d25_display['predicted_round'] = pred_25
d25_display['tier'] = [qb_tier(x)[0] for x in pred_25]
d25_display['Round'] = pd.to_numeric(d25_display['Round'], errors='coerce').astype('Int64')
print('2025 drafted QBs')
display(d25_display.sort_values('predicted_round').reset_index(drop=True))

2025 drafted QBs


,Round,Pick,Player,School,predicted_round,tier
0,6,185.00,Will Howard,Ohio State,1.00,Round 1
1,1,25.00,Jaxson Dart,Mississippi,1.22,Round 1
2,7,227.00,Kurtis Rourke,Indiana,1.85,Round 2
3,5,144.00,Shedeur Sanders,Colorado,2.51,Round 2
4,6,181.00,Kyle McCord,Syracuse,2.65,Round 2
5,2,40.00,Tyler Shough,Louisville,3.30,Round 3
6,6,189.00,Riley Leonard,Notre Dame,3.67,Round 3
7,6,197.00,Graham Mertz,Florida,3.88,Round 4
8,7,231.00,Quinn Ewers,Texas,4.38,Round 4
9,3,94.00,Dillon Gabriel,Oregon,4.63,Round 4


In [9]:
# ── 2026 QB predictions ───────────────────────────────────────────────────
qb_2026 = prepare_qb_df(qb_2026, 2026)
X_26 = scaler.transform(imputer.transform(qb_2026[FEATURES_ALL]))
pred_26 = np.clip(ridge.predict(X_26), 1, 8)

d26_display = qb_2026[['Player', 'School']].copy()
d26_display['Pos'] = 'QB'
d26_display['predicted_round'] = pred_26
d26_display['tier'] = [qb_tier(x)[0] for x in pred_26]
d26_sorted = d26_display.sort_values('predicted_round').reset_index(drop=True)

print(f'2026 QB predictions (n={len(pred_26)})')
display(d26_sorted)

# Save predictions
d26_display[['Player', 'School', 'Pos', 'predicted_round']].to_csv(
    '../data/processed/qb_2026_predictions.csv', index=False
)
print('Saved qb_2026_predictions.csv')

2026 QB predictions (n=16)


,Player,School,Pos,predicted_round,tier
0,Fernando Mendoza,Indiana,QB,1.00,Round 1
1,Diego Pavia,Vanderbilt,QB,2.44,Round 2
2,Joey Aguilar,Tennessee,QB,3.06,Round 3
3,Carson Beck,Miami,QB,3.38,Round 3
4,Ty Simpson,Alabama,QB,3.49,Round 3
5,Drew Allar,Penn State,QB,3.87,Round 4
6,Behren Morton,Texas Tech,QB,3.99,Round 4
7,Joe Fagnano,UConn,QB,4.71,Round 4
8,Luke Altmyer,Illinois,QB,4.72,Round 4
9,Cade Klubnik,Clemson,QB,4.73,Round 4


Saved qb_2026_predictions.csv


In [10]:
# ── Feature coefficients ─────────────────────────────────────────────────
coef_df = pd.DataFrame({'feature': FEATURES_ALL, 'coefficient': ridge.coef_})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
print('Top 20 features by |coefficient|:')
display(coef_df.head(20).reset_index(drop=True))

Top 20 features by |coefficient|:


,feature,coefficient
0,sack_percent,-0.87
1,epa,-0.86
2,pressure_to_sack_rate,0.75
3,RAS,-0.70
4,Height,-0.55
5,ypa,0.55
6,qb_rating,-0.47
7,p4_conference,-0.39
8,btt_rate,-0.34
9,40yd,-0.32
